# Data Generation
Generates synthetic wealth analytics data: users, sessions, events, trades.
Outputs CSVs to GCS `/raw/` layer.

In [ ]:
# ── Install dependencies ──────────────────────────────────
!pip install faker pandas -q


In [ ]:
# ── Imports ───────────────────────────────────────────────
import pandas as pd
import random
import uuid
from datetime import timedelta
from faker import Faker

fake = Faker()
NUM_USERS = 1000
print('✅ Imports ready')


In [ ]:
# ── 1. Users ──────────────────────────────────────────────
users = []
for i in range(NUM_USERS):
    users.append({
        'user_id': i + 1,
        'signup_date': fake.date_between(start_date='-1y', end_date='today'),
        'country': random.choice(['India', 'USA', 'UK']),
        'account_type': random.choice(['retail', 'premium']),
        'risk_profile': random.choice(['low', 'medium', 'high'])
    })
users_df = pd.DataFrame(users)
print(f'✅ Users: {len(users_df)} rows')
users_df.head()


In [ ]:
# ── 2. Sessions ───────────────────────────────────────────
sessions = []
for i in range(3000):
    user_id = random.randint(1, NUM_USERS)
    start_time = fake.date_time_between(start_date='-30d', end_date='now')
    end_time = start_time + timedelta(minutes=random.randint(1, 120))
    sessions.append({
        'session_id': str(uuid.uuid4()),
        'user_id': user_id,
        'session_start': start_time,
        'session_end': end_time,
        'device': random.choice(['mobile', 'web'])
    })
sessions_df = pd.DataFrame(sessions)
print(f'✅ Sessions: {len(sessions_df)} rows')
sessions_df.head()


In [ ]:
# ── 3. Events ─────────────────────────────────────────────
event_types = ['login', 'view_stock', 'search_stock', 'add_watchlist', 'place_order', 'logout']
events = []
for i in range(10000):
    session = sessions_df.sample(1).iloc[0]
    event_time = session['session_start'] + timedelta(
        minutes=random.randint(0, int((session['session_end'] - session['session_start']).seconds / 60))
    )
    events.append({
        'event_id': str(uuid.uuid4()),
        'user_id': session['user_id'],
        'session_id': session['session_id'],
        'event_type': random.choice(event_types),
        'event_time': event_time
    })
events_df = pd.DataFrame(events)
print(f'✅ Events: {len(events_df)} rows')
events_df.head()


In [ ]:
# ── 4. Trades ─────────────────────────────────────────────
stocks = ['AAPL','MSFT','GOOGL','AMZN','TSLA','NVDA','META','NFLX','INTC','AMD',
          'ADBE','CSCO','PEP','AVGO','COST','TMUS','QCOM','TXN','AMAT','INTU',
          'PYPL','SBUX','ISRG','BKNG','GILD','ADP','MDLZ','VRTX','LRCX','MU']

order_events = events_df[events_df['event_type'] == 'place_order']
trades = []
for i in range(len(order_events)):
    event = order_events.iloc[i]
    trades.append({
        'trade_id': str(uuid.uuid4()),
        'user_id': event['user_id'],
        'stock': random.choice(stocks),
        'trade_type': random.choice(['buy', 'sell']),
        'quantity': random.randint(1, 50),
        'price': round(random.uniform(50, 3000), 2),
        'trade_time': event['event_time']
    })
trades_df = pd.DataFrame(trades)
print(f'✅ Trades: {len(trades_df)} rows')
trades_df.head()


In [ ]:
# ── 5. Save CSVs locally ──────────────────────────────────
import os
os.makedirs('/tmp/raw', exist_ok=True)

users_df.to_csv('/tmp/raw/users.csv', index=False)
sessions_df.to_csv('/tmp/raw/sessions.csv', index=False)
events_df.to_csv('/tmp/raw/events.csv', index=False)
trades_df.to_csv('/tmp/raw/trades.csv', index=False)

print('✅ All CSVs saved to /tmp/raw/')
print('   Next: run 02_ingestion/ingest_to_bronze.ipynb')


In [ ]:
# ── 6. Upload raw CSVs to GCS ─────────────────────────────
# Run this only after GCP auth is done in main_pipeline.ipynb
from google.cloud import storage

BUCKET_NAME = 'wealth-analytics-data-lake'
client = storage.Client()
bucket = client.bucket(BUCKET_NAME)

for name in ['users', 'sessions', 'events', 'trades']:
    local = f'/tmp/raw/{name}.csv'
    blob = bucket.blob(f'raw/{name}.csv')
    blob.upload_from_filename(local)
    print(f'  ✅ raw/{name}.csv → GCS')

print('\n✅ Raw layer upload complete')
